# VLM VQA Benchmark — Flood Detection

Zero-shot benchmark of **BLIP-2** and **LLaVA-1.5** on satellite flood imagery.

Three questions, each mapping to an existing U-Net model head or derived ground truth:
- **Q1 (Detection)** → binary yes/no flood presence (`det_true`)
- **Q2 (Severity)** → 4-class flood severity (`severity_bin`)
- **Q3 (Coverage)** → 4-class water coverage from `water_fraction_temporal`

Eval set: **100 samples, 25 per severity bin** (stratified, seed=42). No fine-tuning — pure zero-shot VQA.

In [1]:
# Install extra deps not pre-installed on Kaggle
!pip install -q "transformers>=4.37.0" accelerate sentencepiece


In [1]:
import os, json, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from scipy.stats import kendalltau
from sklearn.metrics import (
    accuracy_score, f1_score, precision_recall_fscore_support,
    confusion_matrix, roc_auc_score,
)

warnings.filterwarnings("ignore", category=rasterio.errors.NotGeoreferencedWarning)

# ── project root (local, git worktree, Kaggle) ─────────────────────────────
_cwd = Path(".").resolve()
_parts = _cwd.parts
if ".claude" in _parts:
    ROOT = Path(*_parts[: _parts.index(".claude")])
else:
    ROOT = _cwd

# ── SEN12FLOOD location ─────────────────────────────────────────────────────
DATA_ROOT_OVERRIDE = None   # set this only if auto-detection fails

_data_candidates = [
    # Kaggle dataset: rhythmroy/sen12flood-flood-detection-dataset
    Path("/kaggle/input/datasets/rhythmroy/sen12flood-flood-detection-dataset/SEN12FLOOD"),
    # Generic Kaggle slug fallbacks
    Path("/kaggle/input/sen12flood-flood-detection-dataset/SEN12FLOOD"),
    Path("/kaggle/input/sen12flood/SEN12FLOOD"),
    Path("/kaggle/input/sen12flood"),
    # Local / Colab
    ROOT / "SEN12FLOOD",
    ROOT / "sen12flood",
    ROOT.parent / "SEN12FLOOD",
    ROOT.parent / "sen12flood",
    Path.home() / "SEN12FLOOD",
    Path("/content/SEN12FLOOD"),
]
# Catch any other Kaggle mount layout automatically
if Path("/kaggle/input").exists():
    for _d in Path("/kaggle/input").rglob("SEN12FLOOD"):
        if _d not in _data_candidates:
            _data_candidates.insert(0, _d)

DATA_ROOT = Path(DATA_ROOT_OVERRIDE) if DATA_ROOT_OVERRIDE else next(
    (p for p in _data_candidates if p.exists()), ROOT / "SEN12FLOOD"
)

# ── proxy_outputs CSVs ──────────────────────────────────────────────────────
_out_candidates = [ROOT / "proxy_outputs"]
if Path("/kaggle/input").exists():
    _out_candidates += [d / "proxy_outputs" for d in Path("/kaggle/input").iterdir()
                        if (d / "proxy_outputs").exists()]

OUT_DIR = next((p for p in _out_candidates if p.exists()), ROOT / "proxy_outputs")

VLM_DIR = (Path("/kaggle/working/proxy_outputs/vlm")
           if Path("/kaggle/working").exists() else ROOT / "proxy_outputs" / "vlm")
CM_DIR  = VLM_DIR / "confusion_matrices"
VLM_DIR.mkdir(parents=True, exist_ok=True)
CM_DIR.mkdir(parents=True, exist_ok=True)

# ── device ──────────────────────────────────────────────────────────────────
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

DTYPE = torch.float16 if DEVICE.type in ("cuda", "mps") else torch.float32

print(f"ROOT     : {ROOT}")
print(f"DATA_ROOT: {DATA_ROOT}  (exists: {DATA_ROOT.exists()})")
print(f"OUT_DIR  : {OUT_DIR}  (exists: {OUT_DIR.exists()})")
print(f"VLM_DIR  : {VLM_DIR}")
print(f"Device   : {DEVICE}  |  dtype: {DTYPE}")
if not DATA_ROOT.exists():
    print("\n⚠️  SEN12FLOOD not found. Set DATA_ROOT_OVERRIDE to your data path.")
if not OUT_DIR.exists():
    print("⚠️  proxy_outputs not found. Run sen12flood_proxy_labels_severity.ipynb first,")
    print("   or add your proxy_outputs folder as a Kaggle dataset data source.")


Device: mps  |  dtype: torch.float16


## Section 1 — Build Evaluation Set

Stratified sample of **25 per severity bin = 100 total** from `severity_manifest.csv`.  
Ground truths derived from existing labels — no human annotation needed.

| Question | Source column | Mapping |
|---|---|---|
| Q1 (Detection) | `severity_bin > 0` | 0 → no-flood, 1 → flood |
| Q2 (Severity)  | `severity_bin`     | 0/1/2/3 → none/mild/moderate/severe |
| Q3 (Coverage)  | `water_fraction_temporal` | <0.10→0, 0.10-0.25→1, 0.25-0.45→2, ≥0.45→3 |

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

manifest_path  = OUT_DIR / "severity_manifest.csv"
severity_path  = OUT_DIR / "proxy_labels_severity.csv"
test_pred_path = OUT_DIR / "cross_region_analysis" / "test_predictions_by_scene.csv"

manifest_df  = pd.read_csv(manifest_path)
severity_df  = pd.read_csv(severity_path)
test_pred_df = pd.read_csv(test_pred_path)

# Normalise region dtype for joining
for df in (manifest_df, severity_df, test_pred_df):
    df["region"] = df["region"].astype(str).str.lstrip("0").replace("", "0")

# Merge water_fraction_temporal into manifest (already in manifest, but ensure present)
if "water_fraction_temporal" not in manifest_df.columns:
    manifest_df = manifest_df.merge(
        severity_df[["region", "scene_id", "water_fraction_temporal"]],
        on=["region", "scene_id"], how="left"
    )

# Drop rows with missing band paths or water fraction
needed_cols = ["s2_b02", "s2_b03", "s2_b04", "severity_bin", "water_fraction_temporal"]
manifest_df = manifest_df.dropna(subset=needed_cols).reset_index(drop=True)

print(f"Total valid rows in manifest: {len(manifest_df)}")
print("Severity bin distribution:")
print(manifest_df["severity_bin"].value_counts().sort_index())

Total valid rows in manifest: 2237
Severity bin distribution:
severity_bin
0    1944
1      81
2      46
3     166
Name: count, dtype: int64


In [3]:
SAMPLES_PER_BIN = 25

# Test regions are ≥ 301 (same split as flood_multitask.ipynb)
TEST_REGION_MIN = 301

def is_test_region(r):
    try:
        return int(float(str(r))) >= TEST_REGION_MIN
    except (ValueError, TypeError):
        return False

def stratified_sample(df, bin_col, n_per_bin, seed=42):
    rng = np.random.default_rng(seed)
    parts = []
    for b in range(4):
        pool = df[df[bin_col] == b].copy()
        if len(pool) == 0:
            print(f"  WARNING: bin {b} has 0 samples")
            continue
        if len(pool) < n_per_bin:
            print(f"  WARNING: bin {b} has only {len(pool)} samples (< {n_per_bin}), taking all")
            parts.append(pool)
            continue
        # Prefer test regions so U-Net baseline overlaps as much as possible
        test_pool  = pool[pool["region"].apply(is_test_region)]
        other_pool = pool[~pool.index.isin(test_pool.index)]
        if len(test_pool) >= n_per_bin:
            idx = rng.choice(len(test_pool), n_per_bin, replace=False)
            selected = test_pool.iloc[idx]
        else:
            need = n_per_bin - len(test_pool)
            fill_idx = rng.choice(len(other_pool), min(need, len(other_pool)), replace=False)
            selected = pd.concat([test_pool, other_pool.iloc[fill_idx]], ignore_index=True)
        parts.append(selected)
    return pd.concat(parts, ignore_index=True)

eval_df = stratified_sample(manifest_df, "severity_bin", SAMPLES_PER_BIN, seed=SEED)

# ── Derive ground truth labels ──────────────────────────────────────────────
def wf_to_coverage_bin(wf):
    if wf < 0.10: return 0
    if wf < 0.25: return 1
    if wf < 0.45: return 2
    return 3

eval_df["q1_gt"] = (eval_df["severity_bin"] > 0).astype(int)
eval_df["q2_gt"] = eval_df["severity_bin"].astype(int)
eval_df["q3_gt"] = eval_df["water_fraction_temporal"].apply(wf_to_coverage_bin)

# ── Join U-Net predictions where available (test set overlap) ───────────────
eval_df = eval_df.merge(
    test_pred_df[["region", "scene_id", "det_pred", "sev_pred", "det_prob"]].rename(
        columns={"det_pred": "unet_det_pred", "sev_pred": "unet_sev_pred"}
    ),
    on=["region", "scene_id"], how="left"
)

eval_df.to_csv(VLM_DIR / "eval_sample_index.csv", index=False)
print(f"Eval set: {len(eval_df)} samples")
print("Ground truth distribution (Q2 = severity):")
print(eval_df["q2_gt"].value_counts().sort_index())
print(f"From test regions (≥{TEST_REGION_MIN}): {eval_df['region'].apply(is_test_region).sum()} / {len(eval_df)}")
print(f"U-Net predictions available for: {eval_df['unet_sev_pred'].notna().sum()} / {len(eval_df)} samples")



Eval set: 100 samples
Ground truth distribution (Q2 = severity):
q2_gt
0    25
1    25
2    25
3    25
Name: count, dtype: int64
U-Net predictions available for: 12 / 100 samples


## Section 2 — Image Loading

Reuses `percentile_stretch()` from `gradcam_severity_pipeline.ipynb` unchanged.

In [4]:
def read_band(path):
    with rasterio.open(str(path)) as src:
        return src.read(1).astype(np.float32)

def percentile_stretch(band, pmin=2, pmax=98):
    lo, hi = np.nanpercentile(band, [pmin, pmax])
    if hi <= lo:
        return np.zeros_like(band, dtype=np.uint8)
    out = np.clip(band, lo, hi)
    out = (out - lo) / (hi - lo)
    return (out * 255).astype(np.uint8)

def load_rgb_pil(b04_path, b03_path, b02_path, size=224):
    """Load Sentinel-2 true-colour RGB (B04=R, B03=G, B02=B) as a PIL Image.

    Paths in severity_manifest.csv are relative to the project root,
    e.g.  SEN12FLOOD/0275/S2_2019-02-25_B04.tif
    ROOT resolves to the project root (worktree-aware), so ROOT / path is correct.
    """
    r = percentile_stretch(read_band(ROOT / b04_path))
    g = percentile_stretch(read_band(ROOT / b03_path))
    b = percentile_stretch(read_band(ROOT / b02_path))
    img = Image.fromarray(np.stack([r, g, b], axis=-1))
    return img.resize((size, size), Image.BILINEAR)

# ── Sanity check ─────────────────────────────────────────────────────────────
if not DATA_ROOT.exists():
    print(f"⚠️  DATA_ROOT not found: {DATA_ROOT}")
    print("   Set DATA_ROOT_OVERRIDE in the paths cell to your SEN12FLOOD directory.")
else:
    row0 = eval_df.iloc[0]
    try:
        img0 = load_rgb_pil(row0.s2_b04, row0.s2_b03, row0.s2_b02)
        plt.figure(figsize=(3, 3))
        plt.imshow(img0)
        plt.axis("off")
        plt.title(f"Region {row0.region} | bin={row0.q2_gt}")
        plt.tight_layout()
        plt.show()
        print("Image loading OK")
    except FileNotFoundError as e:
        print(f"Image load failed: {e}")
        print(f"  ROOT      = {ROOT}")
        print(f"  DATA_ROOT = {DATA_ROOT}")
        print(f"  Band path = {ROOT / row0.s2_b04}")
        print("  → Set DATA_ROOT_OVERRIDE in cell 3 to the folder containing SEN12FLOOD/")


Image load failed: /Users/kritikabansal/computervisionflooddetection/.claude/worktrees/cool-agnesi-4c3abc/SEN12FLOOD/0275/S2_2019-02-25_B04.tif: No such file or directory — check that SEN12FLOOD/ data is present at /Users/kritikabansal/computervisionflooddetection/.claude/worktrees/cool-agnesi-4c3abc/SEN12FLOOD


## Section 3 — Answer Parsers + Prompts

Each parser maps free-form VLM text to an integer label.  
**Unparseable = -1** (excluded from metric computation, counted as parse failures).

In [ ]:
# ── Questions ──────────────────────────────────────────────────────────────
Q1 = "Is there flooding or standing water visible in this satellite image? Answer only: yes or no."
Q2 = "What is the severity of flooding visible in this satellite image? Answer only one word: none, mild, moderate, or severe."
Q3 = "What fraction of this satellite image is covered by water or inundation? Answer only one word: none, low, moderate, or high."

QUESTIONS = {"Q1": Q1, "Q2": Q2, "Q3": Q3}

# ── Parsers ──────────────────────────────────────────────────────────────────
def parse_detection(text):
    t = text.strip().lower()
    # Prioritise explicit negation first to avoid "no flood = yes"
    if any(w in t for w in ["no ", "no.", "none", "dry", "clear", "not flood", "no flood",
                             "doesn't", "does not", "cannot see"]):
        return 0
    if any(w in t for w in ["yes", "flood", "water", "inundation", "submerged", "wet",
                             "standing water", "overflow"]):
        return 1
    if t.startswith("no") and len(t) <= 4:
        return 0
    if t.startswith("yes") and len(t) <= 4:
        return 1
    return -1

def parse_severity(text):
    t = text.strip().lower()
    if "severe" in t:   return 3
    if "moderate" in t: return 2
    if "mild" in t:     return 1
    if "low" in t and "none" not in t: return 1
    if "none" in t or "no flood" in t or "not flood" in t or "no flooding" in t: return 0
    return -1

def parse_coverage(text):
    t = text.strip().lower()
    if "high" in t:     return 3
    if "moderate" in t: return 2
    if "low" in t:      return 1
    if "none" in t or "no water" in t: return 0
    return -1

PARSERS = {"Q1": parse_detection, "Q2": parse_severity, "Q3": parse_coverage}

# ── Quick parser test ─────────────────────────────────────────────────────────
assert parse_detection("Yes, there is flooding.") == 1
assert parse_detection("No flooding visible.") == 0
assert parse_severity("The flooding appears moderate.") == 2
assert parse_severity("None.") == 0
assert parse_coverage("High.") == 3
print("Parser tests passed.")

## Section 4 — BLIP-2 Inference

Model: `Salesforce/blip2-opt-2.7b` (~5 GB). Prompt format: `Question: {q} Answer:`

In [ ]:
from transformers import AutoProcessor, Blip2ForConditionalGeneration

print("Loading BLIP-2 (Salesforce/blip2-opt-2.7b) …")
blip2_processor = AutoProcessor.from_pretrained("Salesforce/blip2-opt-2.7b")
blip2_model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b",
    torch_dtype=DTYPE,
    device_map="auto" if DEVICE.type == "cuda" else None,
)
if DEVICE.type != "cuda":
    blip2_model = blip2_model.to(DEVICE)
blip2_model.eval()
print("BLIP-2 loaded.")

In [ ]:
def run_blip2(image_pil, question, max_new_tokens=30):
    prompt = f"Question: {question} Answer:"
    inputs = blip2_processor(
        images=image_pil, text=prompt, return_tensors="pt"
    ).to(DEVICE, DTYPE)
    with torch.no_grad():
        out = blip2_model.generate(**inputs, max_new_tokens=max_new_tokens)
    full = blip2_processor.decode(out[0], skip_special_tokens=True)
    # Strip prompt echo — BLIP-2 sometimes echoes "Question: ... Answer: <reply>"
    if "Answer:" in full:
        return full.split("Answer:")[-1].strip()
    return full.strip()

records_blip2 = []
n = len(eval_df)
print(f"Running BLIP-2 on {n} samples × 3 questions = {n*3} forward passes …")

for i, row in eval_df.iterrows():
    try:
        img = load_rgb_pil(row.s2_b04, row.s2_b03, row.s2_b02)
    except Exception as e:
        print(f"  [SKIP] sample {i}: image load failed — {e}")
        records_blip2.append({
            "idx": i, "region": row.region, "scene_id": row.scene_id,
            "severity_bin": row.q2_gt,
            "q1_raw": None, "q1_pred": -1,
            "q2_raw": None, "q2_pred": -1,
            "q3_raw": None, "q3_pred": -1,
        })
        continue

    rec = {"idx": i, "region": row.region, "scene_id": row.scene_id, "severity_bin": row.q2_gt}
    for qname, qtext in QUESTIONS.items():
        raw = run_blip2(img, qtext)
        parser = PARSERS[qname]
        rec[f"{qname.lower()}_raw"]  = raw
        rec[f"{qname.lower()}_pred"] = parser(raw)
    records_blip2.append(rec)

    if (i + 1) % 20 == 0:
        print(f"  {i+1}/{n} done")

blip2_df = pd.DataFrame(records_blip2)
blip2_df.to_csv(VLM_DIR / "blip2_raw_answers.csv", index=False)
print(f"Saved {len(blip2_df)} rows → {VLM_DIR / 'blip2_raw_answers.csv'}")
blip2_df.head(3)

## Section 5 — LLaVA-1.5 Inference

Model: `llava-hf/llava-1.5-7b-hf` (~13 GB).  
Set `SKIP_LLAVA = True` if VRAM/RAM is insufficient.

In [ ]:
SKIP_LLAVA = False  # Set True to skip LLaVA if out of memory

if not SKIP_LLAVA:
    from transformers import LlavaForConditionalGeneration, AutoProcessor as LlavaAutoProc
    print("Loading LLaVA-1.5 (llava-hf/llava-1.5-7b-hf) …")
    llava_processor = LlavaAutoProc.from_pretrained("llava-hf/llava-1.5-7b-hf")
    llava_model = LlavaForConditionalGeneration.from_pretrained(
        "llava-hf/llava-1.5-7b-hf",
        torch_dtype=DTYPE,
        device_map="auto" if DEVICE.type == "cuda" else None,
        low_cpu_mem_usage=True,
    )
    if DEVICE.type != "cuda":
        llava_model = llava_model.to(DEVICE)
    llava_model.eval()
    print("LLaVA-1.5 loaded.")
else:
    print("SKIP_LLAVA=True — skipping LLaVA inference.")

In [ ]:
def run_llava(image_pil, question, max_new_tokens=30):
    # LLaVA-1.5 chat template: USER: <image>\n{question}\nASSISTANT:
    prompt = f"USER: <image>\n{question}\nASSISTANT:"
    inputs = llava_processor(
        text=prompt, images=image_pil, return_tensors="pt"
    ).to(DEVICE)
    with torch.no_grad():
        out = llava_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    full = llava_processor.decode(out[0], skip_special_tokens=True)
    if "ASSISTANT:" in full:
        return full.split("ASSISTANT:")[-1].strip()
    return full.strip()

if not SKIP_LLAVA:
    records_llava = []
    n = len(eval_df)
    print(f"Running LLaVA-1.5 on {n} samples × 3 questions = {n*3} forward passes …")

    for i, row in eval_df.iterrows():
        try:
            img = load_rgb_pil(row.s2_b04, row.s2_b03, row.s2_b02)
        except Exception as e:
            print(f"  [SKIP] sample {i}: image load failed — {e}")
            records_llava.append({
                "idx": i, "region": row.region, "scene_id": row.scene_id,
                "severity_bin": row.q2_gt,
                "q1_raw": None, "q1_pred": -1,
                "q2_raw": None, "q2_pred": -1,
                "q3_raw": None, "q3_pred": -1,
            })
            continue

        rec = {"idx": i, "region": row.region, "scene_id": row.scene_id, "severity_bin": row.q2_gt}
        for qname, qtext in QUESTIONS.items():
            raw = run_llava(img, qtext)
            parser = PARSERS[qname]
            rec[f"{qname.lower()}_raw"]  = raw
            rec[f"{qname.lower()}_pred"] = parser(raw)
        records_llava.append(rec)

        if (i + 1) % 20 == 0:
            print(f"  {i+1}/{n} done")

    llava_df = pd.DataFrame(records_llava)
    llava_df.to_csv(VLM_DIR / "llava_raw_answers.csv", index=False)
    print(f"Saved {len(llava_df)} rows → {VLM_DIR / 'llava_raw_answers.csv'}")
    llava_df.head(3)
else:
    llava_df = None
    print("LLaVA skipped.")

## Section 6 — Metrics & Benchmark Results

Metrics computed per **(model × question)**:

| Question | Metrics |
|---|---|
| Q1 Detection (binary) | Accuracy, Precision, Recall, F1, AUC, Parse-fail% |
| Q2 Severity (4-class) | Accuracy, Macro-F1, Per-class F1, Kendall's τ, Bias, Parse-fail% |
| Q3 Coverage (4-class) | Same as Q2 |

**Kendall's τ** (ordinal rank correlation) is the key metric for Q2/Q3 — a model that consistently predicts one bin lower than truth has high τ but low F1.

U-Net baseline included where test-set predictions are available.

In [ ]:
def q1_metrics(df, pred_col, gt_col="q1_gt"):
    valid = df[df[pred_col] != -1].copy()
    parse_fail_pct = 100.0 * (df[pred_col] == -1).sum() / len(df)
    if len(valid) == 0:
        return {"accuracy": float("nan"), "precision": float("nan"),
                "recall": float("nan"), "f1": float("nan"),
                "auc": float("nan"), "parse_fail_pct": parse_fail_pct}
    yt, yp = valid[gt_col].values, valid[pred_col].values
    acc  = accuracy_score(yt, yp)
    prec, rec, f1, _ = precision_recall_fscore_support(yt, yp, average="binary", zero_division=0)
    try:
        auc = roc_auc_score(yt, yp)
    except Exception:
        auc = float("nan")
    return {"accuracy": acc, "precision": prec, "recall": rec,
            "f1": f1, "auc": auc, "parse_fail_pct": parse_fail_pct}


def ordinal_metrics(df, pred_col, gt_col, n_classes=4):
    valid = df[df[pred_col] != -1].copy()
    parse_fail_pct = 100.0 * (df[pred_col] == -1).sum() / len(df)
    if len(valid) == 0:
        base = {"accuracy": float("nan"), "macro_f1": float("nan"),
                "kendall_tau": float("nan"), "bias": float("nan"),
                "parse_fail_pct": parse_fail_pct}
        for b in range(n_classes):
            base[f"f1_bin{b}"] = float("nan")
        return base
    yt = valid[gt_col].values.astype(int)
    yp = valid[pred_col].values.astype(int)
    acc      = accuracy_score(yt, yp)
    macro_f1 = f1_score(yt, yp, average="macro", zero_division=0, labels=list(range(n_classes)))
    pc_f1    = f1_score(yt, yp, average=None, zero_division=0, labels=list(range(n_classes)))
    tau, _   = kendalltau(yt, yp)
    bias     = float(np.mean(yp.astype(float) - yt.astype(float)))
    out = {"accuracy": acc, "macro_f1": macro_f1, "kendall_tau": tau,
           "bias": bias, "parse_fail_pct": parse_fail_pct}
    for b in range(n_classes):
        out[f"f1_bin{b}"] = float(pc_f1[b]) if b < len(pc_f1) else float("nan")
    return out


print("Metric functions defined.")

In [ ]:
# ── Merge answer predictions with ground truth labels ─────────────────────
def attach_gt(pred_df, eval_df):
    return pred_df.merge(
        eval_df[["region", "scene_id", "q1_gt", "q2_gt", "q3_gt"]],
        on=["region", "scene_id"], how="left"
    )

blip2_full = attach_gt(blip2_df, eval_df)
llava_full = attach_gt(llava_df, eval_df) if llava_df is not None else None

# ── Build results table ────────────────────────────────────────────────────
rows = []

for model_name, df in [("BLIP-2", blip2_full), ("LLaVA-1.5", llava_full)]:
    if df is None:
        continue
    m_q1 = q1_metrics(df, "q1_pred", "q1_gt")
    m_q2 = ordinal_metrics(df, "q2_pred", "q2_gt")
    m_q3 = ordinal_metrics(df, "q3_pred", "q3_gt")

    rows.append({"Model": model_name, "Question": "Q1 Detection",
                 "Accuracy": m_q1["accuracy"], "Macro_F1": m_q1["f1"],
                 "Kendall_tau": float("nan"), "AUC": m_q1["auc"],
                 "Bias": float("nan"), "Parse_fail_pct": m_q1["parse_fail_pct"],
                 "F1_bin0": float("nan"), "F1_bin1": float("nan"),
                 "F1_bin2": float("nan"), "F1_bin3": float("nan")})
    for qname, m in [("Q2 Severity", m_q2), ("Q3 Coverage", m_q3)]:
        rows.append({"Model": model_name, "Question": qname,
                     "Accuracy": m["accuracy"], "Macro_F1": m["macro_f1"],
                     "Kendall_tau": m["kendall_tau"], "AUC": float("nan"),
                     "Bias": m["bias"], "Parse_fail_pct": m["parse_fail_pct"],
                     "F1_bin0": m["f1_bin0"], "F1_bin1": m["f1_bin1"],
                     "F1_bin2": m["f1_bin2"], "F1_bin3": m["f1_bin3"]})

# ── U-Net baseline (test-set overlap only) ────────────────────────────────
unet_sub = eval_df.dropna(subset=["unet_sev_pred", "unet_det_pred"]).copy()
unet_sub["unet_det_pred"] = unet_sub["unet_det_pred"].astype(int)
unet_sub["unet_sev_pred"] = unet_sub["unet_sev_pred"].astype(int)
if len(unet_sub) > 0:
    mu_q1 = q1_metrics(unet_sub, "unet_det_pred", "q1_gt")
    mu_q2 = ordinal_metrics(unet_sub, "unet_sev_pred", "q2_gt")
    rows.append({"Model": f"U-Net (n={len(unet_sub)})", "Question": "Q1 Detection",
                 "Accuracy": mu_q1["accuracy"], "Macro_F1": mu_q1["f1"],
                 "Kendall_tau": float("nan"), "AUC": mu_q1["auc"],
                 "Bias": float("nan"), "Parse_fail_pct": 0.0,
                 "F1_bin0": float("nan"), "F1_bin1": float("nan"),
                 "F1_bin2": float("nan"), "F1_bin3": float("nan")})
    rows.append({"Model": f"U-Net (n={len(unet_sub)})", "Question": "Q2 Severity",
                 "Accuracy": mu_q2["accuracy"], "Macro_F1": mu_q2["macro_f1"],
                 "Kendall_tau": mu_q2["kendall_tau"], "AUC": float("nan"),
                 "Bias": mu_q2["bias"], "Parse_fail_pct": 0.0,
                 "F1_bin0": mu_q2["f1_bin0"], "F1_bin1": mu_q2["f1_bin1"],
                 "F1_bin2": mu_q2["f1_bin2"], "F1_bin3": mu_q2["f1_bin3"]})

results_df = pd.DataFrame(rows)
results_df.to_csv(VLM_DIR / "benchmark_results.csv", index=False)
print("benchmark_results.csv saved.")

In [ ]:
display_cols = ["Model", "Question", "Accuracy", "Macro_F1", "Kendall_tau", "AUC",
                "Bias", "Parse_fail_pct"]
fmt = {c: "{:.3f}" for c in display_cols if c not in ("Model", "Question")}

print("=" * 90)
print(f"{'Model':<22} {'Question':<18} {'Acc':>6} {'MacroF1':>8} {'τ':>6} {'AUC':>6} {'Bias':>6} {'Fail%':>6}")
print("-" * 90)
for _, r in results_df.iterrows():
    def fmt_val(v):
        return f"{v:6.3f}" if pd.notna(v) and v == v else "   —  "
    print(f"{str(r.Model):<22} {str(r.Question):<18}"
          f" {fmt_val(r.Accuracy)} {fmt_val(r.Macro_F1)} {fmt_val(r.Kendall_tau)}"
          f" {fmt_val(r.AUC)} {fmt_val(r.Bias)} {fmt_val(r.Parse_fail_pct)}")
print("=" * 90)

print("\nPer-class F1 (Q2 Severity & Q3 Coverage):")
pc_cols = ["Model", "Question", "F1_bin0", "F1_bin1", "F1_bin2", "F1_bin3"]
pc_df = results_df[results_df["Question"].isin(["Q2 Severity", "Q3 Coverage"])][pc_cols]
print(pc_df.to_string(index=False, float_format="{:.3f}".format))

In [ ]:
def plot_confusion_matrix(ax, df, pred_col, gt_col, labels, title):
    valid = df[df[pred_col] != -1]
    if len(valid) == 0:
        ax.set_title(f"{title}\n(no data)")
        ax.axis("off")
        return
    cm = confusion_matrix(valid[gt_col].astype(int), valid[pred_col].astype(int),
                          labels=list(range(len(labels))))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=labels, yticklabels=labels, cbar=False)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title, fontsize=9)

label_q1  = ["no-flood", "flood"]
labels_q2 = ["none", "mild", "mod.", "severe"]
labels_q3 = ["none", "low", "mod.", "high"]

models_data = [(name, df) for name, df in
               [("BLIP-2", blip2_full), ("LLaVA-1.5", llava_full)] if df is not None]
n_models = len(models_data)

fig, axes = plt.subplots(n_models, 3, figsize=(13, 4 * n_models))
if n_models == 1:
    axes = axes[np.newaxis, :]  # ensure 2-D indexing

for row_i, (model_name, df) in enumerate(models_data):
    plot_confusion_matrix(axes[row_i, 0], df, "q1_pred", "q1_gt", label_q1,
                          f"{model_name} — Q1 Detection")
    plot_confusion_matrix(axes[row_i, 1], df, "q2_pred", "q2_gt", labels_q2,
                          f"{model_name} — Q2 Severity")
    plot_confusion_matrix(axes[row_i, 2], df, "q3_pred", "q3_gt", labels_q3,
                          f"{model_name} — Q3 Coverage")

plt.suptitle("Confusion Matrices — VLM VQA Benchmark", fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(CM_DIR / "all_confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {CM_DIR / 'all_confusion_matrices.png'}")

In [ ]:
bin_labels = ["Bin 0\n(none)", "Bin 1\n(mild)", "Bin 2\n(mod.)", "Bin 3\n(severe)"]
x = np.arange(len(bin_labels))
width = 0.35

for qname, gt_col, pred_col_suffix in [("Q2 Severity", "q2_gt", "q2_pred"),
                                        ("Q3 Coverage",  "q3_gt", "q3_pred")]:
    fig, ax = plt.subplots(figsize=(8, 4))
    colour_cycle = ["#4C72B0", "#DD8452"]
    offsets = [-width/2, width/2]

    for ci, (model_name, df) in enumerate(models_data):
        valid = df[df[pred_col_suffix] != -1]
        pc = f1_score(valid[gt_col].astype(int), valid[pred_col_suffix].astype(int),
                      average=None, zero_division=0, labels=[0, 1, 2, 3])
        pc = [float(pc[b]) if b < len(pc) else 0.0 for b in range(4)]
        bars = ax.bar(x + offsets[ci], pc, width, label=model_name,
                      color=colour_cycle[ci], alpha=0.85)
        for bar, val in zip(bars, pc):
            if val > 0.02:
                ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                        f"{val:.2f}", ha="center", va="bottom", fontsize=8)

    ax.set_xticks(x)
    ax.set_xticklabels(bin_labels)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("F1 Score")
    ax.set_title(f"Per-class F1 — {qname}")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    fname = CM_DIR / f"per_class_f1_{qname.lower().replace(' ', '_')}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {fname}")

print("\nAll outputs written to:", VLM_DIR)